In [ ]:
!pip install -q esm

In [ ]:
import pandas as pd
import torch
import os
import numpy as np
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

In [ ]:
columns = [
    "pdb_id",
    "receptor_chain",
    "resolution",
    "binding_site_code",
    "ligand_id",
    "ligand_chain",
    "ligand_serial_number",
    "binding_residues_pdb",
    "binding_residues_seq",
    "catalytic_residues_pdb",
    "catalytic_residues_seq",
    "ec_number",
    "go_terms",
    "binding_affinity_manual",
    "binding_affinity_moad",
    "binding_affinity_pdbbind",
    "binding_affinity_bindingdb",
    "uniprot_id",
    "pubmed_id",
    "ligand_residue_seq_number",
    "sequence"
]

df = pd.read_csv("/kaggle/input/datasets/yangxinzhandu/biolip-nr-txt-gz/BioLiP_nr.txt", sep="\t", names=columns, header=None)

df = df[['pdb_id', 'receptor_chain', 'resolution', 'ligand_id', 'binding_residues_seq','sequence']]

print(df.shape)
print(df.ligand_id.value_counts())
df.head()


In [ ]:
df = df[df['binding_residues_seq'].notna()]
df = df[df['ligand_id'].notna()]

to_remove = {
    # Metal ions
    "ZN", "CA", "MG", "MN", "FE", "FE2", "CU", "CU1", "CO", "NI", "CD", "HG", "MN3",
    "CL", "AG", "LA", "YB", "RB", "PB", "SE", "AU", "BR", "IOD",
    # Nucleic acids
    "RNA", "DNA", "PEPTIDE",
    # Crystallization artifacts
    "SO4", "GOL", "EDO", "PO4", "ACT", "BME", "MPD", "FMT", "ACE", "DMS", "PGE",
    "EOH", "TRS", "IMD", "PGW", "C8E", "C2E",
    # Free amino acids
    "GLY", "ALA", "VAL", "LEU", "ILE", "PRO", "PHE", "TRP", "SER", "THR", "CYS",
    "MET", "ASN", "GLN", "ASP", "GLU", "LYS", "ARG", "HIS", "TYR"
}

df_cleaned = df[~df["ligand_id"].str.upper().isin(to_remove)]
df_cleaned.reset_index(drop= True, inplace=True)
df_cleaned

In [ ]:
df_cleaned['sequence_length (aa)'] = df_cleaned['sequence'].apply(lambda x: len(x))
df_cleaned['binding_residues_seq_length'] = df_cleaned['binding_residues_seq'].apply(lambda x: len(x.split()))
df_cleaned = df_cleaned[df_cleaned['resolution'] <= 2]
df_cleaned = df_cleaned[(df_cleaned['sequence_length (aa)'] >= 150) & (df_cleaned['sequence_length (aa)'] <= 800)]
df_cleaned = df_cleaned.drop_duplicates(['pdb_id','ligand_id'])
df_cleaned = df_cleaned[df_cleaned['binding_residues_seq_length'] >= 8]
df_cleaned.reset_index(drop=True, inplace=True)
df_cleaned

In [ ]:
rows = []

for index, row in df_cleaned.iterrows():
  binding_residues = set()

  for binding_residue in row['binding_residues_seq'].split():
    position = int(''.join(filter(str.isdigit, binding_residue)))
    binding_residues.add(position)

  for i, residue in enumerate(row['sequence']):
    rows.append({
        'pdb_id': row['pdb_id'],
        'receptor_chain': row['receptor_chain'],
        'ligand_id': row['ligand_id'],
        'position': i+1,
        'residue': residue,
        'isbinding': 1 if i+1 in binding_residues else 0,
        })

residue_df = pd.DataFrame(rows)

In [ ]:
residue_df

In [ ]:
y = residue_df['isbinding'].value_counts()
y

In [ ]:
client = ESMC.from_pretrained("esmc_300m").to('cuda')

In [ ]:
os.makedirs("embeddings_proteins", exist_ok=True)

for _, entry in df_cleaned.iterrows():
    pdb_id    = entry["pdb_id"]
    chain     = entry["receptor_chain"]
    ligand    = entry["ligand_id"]

    unique_id = f"{pdb_id}_{chain}_{ligand}"
    save_path = f"embeddings_proteins/{unique_id}.npy"

    if os.path.exists(save_path):
        continue

    seq = entry["sequence"]
    protein = ESMProtein(sequence=seq)
    protein_tensor = client.encode(protein)

    with torch.no_grad():
        output = client.logits(
            protein_tensor,
            LogitsConfig(sequence=True, return_embeddings=True)
        )

    embeddings = output.embeddings.squeeze(0).half().cpu().numpy()[1:-1, :]
    np.save(save_path, embeddings)

    del output, embeddings
    torch.cuda.empty_cache()

print("All embeddings saved to disk")

In [ ]:
EMBEDDINGS_DIR = "embeddings_proteins"
OUTPUT_DIR = "embedding_chunks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for _, entry in df_cleaned.iterrows():
    pdb_id    = entry["pdb_id"]
    chain     = entry["receptor_chain"]
    ligand    = entry["ligand_id"]
    unique_id = f"{pdb_id}_{chain}_{ligand}"

    chunk_path = f"{OUTPUT_DIR}/{unique_id}.parquet"
    if os.path.exists(chunk_path):
        continue

    embeddings = np.load(f"{EMBEDDINGS_DIR}/{unique_id}.npy")

    meta = pd.DataFrame({
        "pdb_id":         pdb_id,
        "receptor_chain": chain,
        "ligand_id":      ligand,
        "position":       np.arange(1, len(embeddings) + 1)
    })

    esm_cols = pd.DataFrame(embeddings, columns=[f"esm_{i}" for i in range(embeddings.shape[1])])

    chunk_df = pd.concat([meta, esm_cols], axis=1)
    chunk_df.to_parquet(chunk_path, index=False)

    del embeddings, chunk_df, meta, esm_cols

print("All chunks saved")

In [ ]:
embedding_df = pd.read_parquet(OUTPUT_DIR)
print(embedding_df.shape)
embedding_df

In [ ]:
df_merge = residue_df.merge(
    embedding_df,
    on=["pdb_id", "receptor_chain", "ligand_id", "position"],
    how="inner"
)

In [ ]:
df_merge = df_merge.drop(columns=['receptor_chain', 'ligand_id'])
df_merge

In [ ]:
df_merge.to_parquet(
    "model_dataset_cleaned_V2.parquet",
    engine="pyarrow",
    compression="zstd"
)